# Publisher ward names update — 12 November 2026
Generation cells include the official 244-ID name lookup and updated dashboard/API. Outputs have been cleared; rerun to validate in your environment. Original report fields and investigation text remain historical; current publisher names do not verify geographic boundaries. See `docs/geography_names.md`.

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/NammaFlow_phase9")

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Narasimhan15/NammaFlow.git", str(repo)],
        check=True,
    )

print("Repository commit:")
subprocess.run(["git", "-C", str(repo), "log", "-1", "--oneline"], check=True)

print("\nApplication files:")
for folder in ["app", "api", "dashboard", "src", "frontend", "backend"]:
    path = repo / folder
    if path.exists():
        files = sorted(p for p in path.rglob("*") if p.is_file())
        print(f"\n{folder}/")
        for file in files[:50]:
            print(" ", file.relative_to(repo))
        if not files:
            print("  (empty)")

print("\nRoot files:")
for path in sorted(repo.iterdir()):
    if path.is_file():
        print(" ", path.name)

In [ ]:
result = subprocess.run(
    ["git", "-C", str(repo), "log", "-1", "--oneline"],
    capture_output=True, text=True, check=True
)
print("Current commit:", result.stdout.strip())

print("\nRepository folders:")
for path in sorted(repo.iterdir()):
    if path.is_dir() and path.name != ".git":
        print(" ", path.name)

print("\nPhase 7 and Phase 8 report files:")
for folder in [
    "reports/ward_level_intelligence",
    "reports/decision_recommendation_engine",
]:
    path = repo / folder
    print(f"\n{folder}:")
    if path.exists():
        for file in sorted(path.rglob("*")):
            if file.is_file():
                print(" ", file.relative_to(repo))
    else:
        print("  Folder not found")

print("\nOther investigation report locations:")
for path in sorted(repo.rglob("investigation_suggestions.csv")):
    print(" ", path.relative_to(repo))

In [ ]:
import pandas as pd

profiles_path = repo / "reports/ward_level_intelligence/ward_profiles.csv"
suggestions_path = repo / "reports/decision_recommendation/investigation_suggestions.csv"

profiles = pd.read_csv(profiles_path)
suggestions = pd.read_csv(suggestions_path)

for name, df in [
    ("Ward profiles", profiles),
    ("Investigation suggestions", suggestions),
]:
    print(f"\n{name}: {df.shape[0]} rows × {df.shape[1]} columns")
    print("Columns:", df.columns.tolist())
    display(df.head(3))

In [ ]:
assert profiles["ward_num"].notna().all()
assert profiles["ward_num"].is_unique
assert suggestions[["ward_num", "rule_id"]].notna().all().all()
assert not suggestions.duplicated(["ward_num", "rule_id"]).any()
assert set(suggestions["ward_num"]).issubset(set(profiles["ward_num"]))

print("Identifier and relationship checks passed.")
print("\nSuggestions per rule:")
print(suggestions.groupby("rule_id").size().to_string())

print("\nRecords with suggestions:", suggestions["ward_num"].nunique())

print("\nSuggestion columns:")
print("\n".join(suggestions.columns))

print("\nProfile columns:")
print("\n".join(profiles.columns))

# Phase 9 — NammaFlow Dashboard and API

## Objective
Build an interactive application and API using the reviewed ward
profiles and investigation suggestions.

## Application features
1. Overview: searches, bookings, completed rides, and aggregate funnel rates.
2. Record explorer: filter and inspect all 244 records.
3. Investigation explorer: filter suggestions by record and rule.
4. Model context: display exploratory clusters, anomaly scores,
   and reasons for missing model outputs.
5. Data limitations: explain eligibility, geographic uncertainty,
   and interpretation constraints.

## API scope
Provide health, summary, record-list, record-detail, and
investigation endpoints using the same data-loading logic as the dashboard.

## Interpretation requirements
- Preserve all records, including zero-booking records and w_0.
- Keep undefined rates missing; display them as unavailable.
- Calculate aggregate rates from summed counts.
- Treat rule triggers as investigation suggestions, not causes or priorities.
- Treat clusters and anomaly scores as exploratory context.
- Geographic mappings remain unverified; maps are deferred.
- These reports describe historical data and do not establish forecasting accuracy.

## Planned implementation
- Streamlit dashboard
- FastAPI API
- Shared Python data-loading and validation module
- Colab notebook for setup, checks, and packaging

**Creating the shared data loading modules. Both dashboard and API will use it to load and validate the reports.**

### What Is the Shared Data Module?
The shared data module (`nammaflow/data.py`) contains reusable functions
used by both the dashboard and API. Keeping this logic in one place
avoids duplicate code and keeps their results consistent.

It loads the ward profiles and investigation suggestions, checks required
columns, identifiers, relationships, and count values, and calculates
summary totals and aggregate rates. It preserves all records and missing
values, returning unavailable rates when a denominator is zero.

In [ ]:
shared_dir = repo / "nammaflow"
shared_dir.mkdir(exist_ok=True)
(shared_dir / "__init__.py").touch()
module_code = 'import json\nfrom pathlib import Path\nimport pandas as pd\n\nDEFAULT_ROOT = Path(__file__).resolve().parents[1]\n\nPROFILE_COLUMNS = [\n    "ward_num", "ward_searches", "ward_bookings",\n    "ward_completed_rides", "ward_search_to_booking_rate",\n    "ward_booking_completion_rate", "has_model_results",\n    "exclusion_reason", "exploratory_cluster", "anomaly_score",\n    "geographic_mapping_status",\n]\n\nSUGGESTION_COLUMNS = [\n    "ward_num", "rule_id", "suggested_investigation", "limitations",\n]\n\n\ndef _unique_keys(pairs):\n    """Reject ambiguous JSON mappings rather than silently keep the last name."""\n    result = {}\n    for key, value in pairs:\n        if key in result:\n            raise ValueError(f"Duplicate ward lookup key: {key}")\n        result[key] = value\n    return result\n\n\ndef load_data(root=None):\n    """Load reports without replacing missing values or excluding records."""\n    root = Path(root) if root is not None else DEFAULT_ROOT\n\n    profiles = pd.read_csv(\n        root / "reports/ward_level_intelligence/ward_profiles.csv",\n        dtype={"ward_num": "string"},\n    )\n    suggestions = pd.read_csv(\n        root / "reports/decision_recommendation/investigation_suggestions.csv",\n        dtype={"ward_num": "string", "rule_id": "string"},\n    )\n\n    for name, frame, required in [\n        ("profiles", profiles, PROFILE_COLUMNS),\n        ("suggestions", suggestions, SUGGESTION_COLUMNS),\n    ]:\n        missing = sorted(set(required) - set(frame.columns))\n        if missing:\n            raise ValueError(f"{name}: missing columns {missing}")\n\n    if profiles.empty:\n        raise ValueError("Profiles report is empty.")\n\n    for name, frame in [("profiles", profiles), ("suggestions", suggestions)]:\n        ids = frame["ward_num"]\n        if ids.isna().any() or ids.str.strip().eq("").any():\n            raise ValueError(f"{name}: missing or blank record identifiers.")\n\n    if profiles["ward_num"].duplicated().any():\n        raise ValueError("Profiles contain duplicate record identifiers.")\n\n    if suggestions["rule_id"].isna().any():\n        raise ValueError("Suggestions contain missing rule identifiers.")\n\n    if suggestions.duplicated(["ward_num", "rule_id"]).any():\n        raise ValueError("Duplicate record-rule suggestions found.")\n\n    if not suggestions["ward_num"].isin(profiles["ward_num"]).all():\n        raise ValueError("Suggestions reference unknown records.")\n\n    counts = ["ward_searches", "ward_bookings", "ward_completed_rides"]\n    for column in counts:\n        values = pd.to_numeric(profiles[column], errors="raise")\n        if (\n            values.isna().any()\n            or not values.between(0, float("inf"), inclusive="left").all()\n            or values.mod(1).ne(0).any()\n        ):\n            raise ValueError(f"{column}: expected finite nonnegative counts.")\n\n    lookup_path = root / "data/reference/ward_names.json"\n    with lookup_path.open(encoding="utf-8") as source:\n        metadata = json.load(source, object_pairs_hook=_unique_keys)\n    names = metadata.get("ward_names")\n    if not isinstance(names, dict) or not names or any(\n        not isinstance(key, str) or not key.strip()\n        or not isinstance(value, str) or not value.strip()\n        for key, value in names.items()\n    ):\n        raise ValueError("Ward lookup requires nonblank string identifiers and names.")\n\n    for frame in (profiles, suggestions):\n        frame["ward_name"] = frame["ward_num"].map(names).astype("string")\n        frame["ward_name_status"] = frame["ward_name"].notna().map(\n            {True: "publisher_label_verified", False: "unmapped"}\n        )\n        frame["display_name"] = (\n            frame["ward_name"].fillna("Unknown area") + " (" + frame["ward_num"] + ")"\n        )\n        # Put readable labels first, retaining all original report fields and order.\n        labels = ["ward_name", "display_name", "ward_name_status"]\n        ordered = labels + [column for column in frame.columns if column not in labels]\n        if frame is profiles:\n            profiles = frame[ordered]\n        else:\n            suggestions = frame[ordered]\n\n    return profiles, suggestions\n\n'
(shared_dir / "data.py").write_text(module_code)
reference_dir = repo / "data/reference"
reference_dir.mkdir(parents=True, exist_ok=True)
(reference_dir / "ward_names.json").write_text('{\n  "source_url": "https://nammayatri.in/chunks/HomeLayout.bs.aa195fb1.js",\n  "source_function": "getBangaloreWardName",\n  "retrieved_date": "2026-11-12",\n  "source_sha256": "85ebcfca68907473ad9446c83ba283ba5c7c35b433fe7713b624983ab7b0ebc2",\n  "scope": "Labels from the current official Namma Yatri dashboard; not certification of current administrative boundaries or historical boundary equivalence.",\n  "dataset_id_count": 244,\n  "matched_id_count": 244,\n  "ward_names": {\n    "b_1": "Kempegowda Ward",\n    "b_10": "Kempapura",\n    "b_100": "Banasavadi",\n    "b_101": "Kammanahalli",\n    "b_102": "Lingarajapura",\n    "b_103": "Maruthi Seva Nagar",\n    "b_104": "Kadugodi",\n    "b_105": "Belathur",\n    "b_106": "Hudi",\n    "b_107": "Garudachar Playa",\n    "b_108": "Dodda Nekkundi",\n    "b_109": "AECS Layout",\n    "b_11": "Byatarayanapura",\n    "b_110": "Whitefield",\n    "b_111": "Hagadur",\n    "b_112": "Varthuru",\n    "b_113": "Munnekollala",\n    "b_114": "Marathahalli",\n    "b_115": "Bellanduru",\n    "b_116": "Doddakanahalli",\n    "b_117": "C V Raman Nagar",\n    "b_118": "Lal Bahadur Nagar",\n    "b_119": "New Bayappanahalli",\n    "b_12": "Kodigehalli",\n    "b_120": "Hoysala Nagar",\n    "b_121": "Old Thippasandra",\n    "b_122": "New Thippasandra",\n    "b_123": "Jalakanteshwara Nagara",\n    "b_124": "Jeevanbhima Nagar",\n    "b_125": "Konena Agrahara",\n    "b_126": "Ramaswamy Palya",\n    "b_127": "Jayamahal",\n    "b_128": "Vasanth Nagar",\n    "b_129": "Sampangiram Nagar",\n    "b_13": "Dodda Bommasandra",\n    "b_130": "Bharathi Nagar",\n    "b_131": "Ulsoor",\n    "b_132": "Dattatreya Temple",\n    "b_133": "Gandhinagar",\n    "b_134": "Subhash Nagar",\n    "b_135": "Okalipuram",\n    "b_136": "Binnipete",\n    "b_137": "Cottonpete",\n    "b_138": "Chickpete",\n    "b_139": "Dayananda Nagar",\n    "b_14": "Vidyaranyapura",\n    "b_140": "Prakash Nagar",\n    "b_141": "Rajaji Nagar",\n    "b_142": "Sriramamandir",\n    "b_143": "Shivanagara",\n    "b_144": "Basaveshwara Nagar",\n    "b_145": "Kamakshipalya",\n    "b_146": "Dr. Raj Kumar Ward",\n    "b_147": "Agrahara Dasarahalli",\n    "b_148": "Govindaraja Nagar",\n    "b_149": "Kaveripura",\n    "b_15": "Kuvempunagar",\n    "b_150": "Marenahalli",\n    "b_151": "Maruthi Mandir ward",\n    "b_152": "Mudalapalya",\n    "b_153": "Nagarabhavi",\n    "b_154": "Chandra Layout",\n    "b_155": "Nayandahalli",\n    "b_156": "Kempapura Agrahara",\n    "b_157": "Vijayanagar",\n    "b_158": "Hosahalli",\n    "b_159": "Hampi Nagar",\n    "b_16": "Kammagondanahalli",\n    "b_160": "Bapuji Nagar",\n    "b_161": "Attiguppe",\n    "b_162": "Gali Anjenaya Temple ward",\n    "b_163": "Veerabhadranagar",\n    "b_164": "Avalahalli",\n    "b_165": "Chamrajapet",\n    "b_166": "Chalavadipalya",\n    "b_167": "Jagajivanaram Nagar",\n    "b_168": "Padarayanapura",\n    "b_169": "Devaraj Urs Nagar",\n    "b_17": "Shettihalli",\n    "b_170": "Azad Nagar",\n    "b_171": "Sudham Nagara",\n    "b_172": "Dharmaraya Swamy Temple Ward",\n    "b_173": "Sunkenahalli",\n    "b_174": "Vishveshwara Puram",\n    "b_175": "Ashoka Pillar",\n    "b_176": "Someshwara Nagar",\n    "b_177": "Hombegowda Nagara",\n    "b_178": "Domlur",\n    "b_179": "Jogupalya",\n    "b_18": "Bagalakunte",\n    "b_180": "Agaram",\n    "b_181": "Shantala Nagar",\n    "b_182": "Shanthi Nagar",\n    "b_183": "Neelasandra",\n    "b_184": "Vannarapete",\n    "b_185": "Ejipura",\n    "b_186": "Koramangala",\n    "b_187": "Adugodi",\n    "b_188": "Lakkasandra",\n    "b_189": "Suddagunte Palya",\n    "b_19": "Defence Colony",\n    "b_190": "Madivala",\n    "b_191": "Jakkasandra",\n    "b_192": "BTM Layout",\n    "b_193": "N S Palya",\n    "b_194": "Gurappanapalya",\n    "b_195": "Tilak Nagar",\n    "b_196": "Byrasandra",\n    "b_197": "Shakambari Nagar",\n    "b_198": "J P Nagar",\n    "b_199": "Sarakki",\n    "b_2": "Chowdeswari Ward",\n    "b_20": "Mallasandra",\n    "b_200": "Yediyur",\n    "b_201": "Umamaheshwara Ward",\n    "b_202": "Ganesh Mandir ward",\n    "b_203": "Banashankari Temple ward",\n    "b_204": "Kumaraswamy Layout",\n    "b_205": "Vikram Nagar",\n    "b_206": "Padmanabha Nagar",\n    "b_207": "Kamakya Nagar",\n    "b_208": "Deen Dayalu Ward",\n    "b_209": "Hosakerehalli",\n    "b_21": "T Dasarahalli",\n    "b_210": "Basavanagudi",\n    "b_211": "Hanumanth Nagar",\n    "b_212": "Srinivasa Nagar",\n    "b_213": "Srinagar",\n    "b_214": "Girinagar",\n    "b_215": "Katriguppe",\n    "b_216": "Vidyapeeta ward",\n    "b_217": "Uttarahalli",\n    "b_218": "Subramanyapura",\n    "b_219": "Vasanthpura",\n    "b_22": "Chokkasandra",\n    "b_220": "Yelachenahalli",\n    "b_221": "Konanakunte",\n    "b_222": "RBI layout",\n    "b_223": "Chunchaghatta",\n    "b_224": "Anjanapura",\n    "b_225": "Gottigere",\n    "b_226": "Kalena Agrahara",\n    "b_227": "Begur",\n    "b_228": "Naganathapura",\n    "b_229": "Ibluru",\n    "b_23": "Nelagadderanahalli",\n    "b_230": "Agara",\n    "b_231": "Mangammanapalya",\n    "b_232": "HSR - Singasandra",\n    "b_233": "Rupenaagrahara",\n    "b_234": "Hongasandra",\n    "b_235": "Bommanahalli",\n    "b_236": "Devarachikkanahalli",\n    "b_237": "Bilekhalli",\n    "b_238": "Arakere",\n    "b_239": "Hulimavu",\n    "b_24": "Rajagopal Nagar",\n    "b_240": "Vinayakanagar",\n    "b_241": "Puttenahalli - Sarakki Lake",\n    "b_242": "Jaraganahalli",\n    "b_243": "Kudlu",\n    "b_25": "Rajeshwari Nagar",\n    "b_26": "Hegganahalli",\n    "b_27": "Sunkadakatte",\n    "b_28": "Dodda Bidarakallu",\n    "b_29": "Vidyamanyanagar",\n    "b_3": "Someshwara Ward",\n    "b_30": "Herohalli",\n    "b_31": "Doddagollarahatti",\n    "b_32": "Ullal",\n    "b_33": "Kengeri",\n    "b_34": "Bande Mutt",\n    "b_35": "Hemmigepura",\n    "b_36": "Chatrapati Shivaji",\n    "b_37": "Chanakya",\n    "b_38": "J P Park",\n    "b_39": "Kanneshwara Rama",\n    "b_4": "Atturu Layout",\n    "b_40": "Veeramadakari",\n    "b_41": "Peenya",\n    "b_42": "Lakshmi Devi Nagar",\n    "b_43": "Ranadheera Kanteerava",\n    "b_44": "Veera Sindhura Lakshamana",\n    "b_45": "Vijayanagara Krishnadevaraya",\n    "b_46": "Sir M. Vishweshwaraiah",\n    "b_47": "Nalvadi Krishnaraja Wadior Park",\n    "b_48": "Jnana Bharathi",\n    "b_49": "Rajarajeshwari Nagar",\n    "b_5": "Yelahanka Satellite Town",\n    "b_50": "Marappana Palya",\n    "b_51": "Nagapura",\n    "b_52": "Mahalakshimpuram",\n    "b_53": "Nandini Layout",\n    "b_54": "Jai Maruthinagara",\n    "b_55": "Puneet Rajkumar",\n    "b_56": "Shankar Matt",\n    "b_57": "Shakthi Ganapathi Nagar",\n    "b_58": "Vrisabhavathi Nagar",\n    "b_59": "Mattikere",\n    "b_6": "Kogilu",\n    "b_60": "Aramane Nagara",\n    "b_61": "Malleswaram",\n    "b_62": "Subramanya Nagar",\n    "b_63": "Gayithri Nagar",\n    "b_64": "Kadu Malleshwara",\n    "b_65": "Rajamahal Guttahalli",\n    "b_66": "Radhakrishna Temple Ward",\n    "b_67": "Sanjaya Nagar",\n    "b_68": "Vishwanath Nagenahalli",\n    "b_69": "Manorayanapalya",\n    "b_7": "Thanisandra",\n    "b_70": "Hebbala",\n    "b_71": "Chamundi Nagara",\n    "b_72": "Ganga Nagar",\n    "b_73": "Jayachamarajendra  Nagar",\n    "b_74": "Kaval Bairasandra",\n    "b_75": "Kushal Nagar",\n    "b_76": "Muneshwara Nagar",\n    "b_77": "Devara Jeevanahalli",\n    "b_78": "S K Garden",\n    "b_79": "Sagayarapuram",\n    "b_8": "Jakkuru",\n    "b_80": "Pulikeshinagar",\n    "b_81": "Horamavu",\n    "b_82": "Babusab Palya",\n    "b_83": "Kalkere",\n    "b_84": "Ramamurthy Nagara",\n    "b_85": "Vijinapura",\n    "b_86": "K R Puram",\n    "b_87": "Medahalli",\n    "b_88": "Basavanapura",\n    "b_89": "Devasandra",\n    "b_9": "Amrutahalli",\n    "b_90": "Mahadevapura",\n    "b_91": "A Narayanapura",\n    "b_92": "Vijnana Nagar",\n    "b_93": "HAL Airport",\n    "b_94": "Hennur",\n    "b_95": "Nagavara",\n    "b_96": "Kadugondanahalli",\n    "b_97": "Venkateshpura",\n    "b_98": "Kacharkanahalli",\n    "b_99": "HRBR Layout",\n    "w_0": "Other Wards"\n  }\n}\n', encoding="utf-8")

import sys
import importlib
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
importlib.invalidate_caches()
from nammaflow import data
importlib.reload(data)
app_profiles, app_suggestions = data.load_data(repo)
pd.testing.assert_frame_equal(app_profiles[profiles.columns], profiles.astype({"ward_num": "string"}))
pd.testing.assert_frame_equal(app_suggestions[suggestions.columns], suggestions.astype({"ward_num": "string", "rule_id": "string"}))
assert app_profiles["ward_name"].notna().all()
print("All original values preserved; publisher names added.")


### Shared Summary Calculations
Calculates total searches, bookings, completed rides, and investigation counts. Aggregate rates use summed counts; zero denominators return unavailable values. All records, including `w_0`, remain included. Checks verify aggregate-rate calculations and zero-denominator handling.

In [ ]:
summary_code = '''
def build_summary(profiles, suggestions):
    """Summarize the supplied records and their matching suggestions."""
    relevant = suggestions[
        suggestions["ward_num"].isin(profiles["ward_num"])
    ]

    searches = int(profiles["ward_searches"].sum())
    bookings = int(profiles["ward_bookings"].sum())
    completed = int(profiles["ward_completed_rides"].sum())

    def safe_rate(numerator, denominator):
        return numerator / denominator if denominator else None

    return {
        "record_count": int(len(profiles)),
        "searches": searches,
        "bookings": bookings,
        "completed_rides": completed,
        "search_to_booking_rate": safe_rate(bookings, searches),
        "booking_completion_rate": safe_rate(completed, bookings),
        "search_to_completion_rate": safe_rate(completed, searches),
        "suggestion_count": int(len(relevant)),
        "records_with_suggestions": int(relevant["ward_num"].nunique()),
        "suggestions_by_rule": {
            str(rule): int(count)
            for rule, count in relevant.groupby("rule_id").size().items()
        },
    }
'''

# Replace this section on reruns instead of adding duplicate functions.
module_path = shared_dir / "data.py"
marker = "# --- Summary functions ---"
base_code = module_path.read_text(encoding="utf-8").split(marker)[0]
module_path.write_text(
    base_code.rstrip() + "\n\n" + marker + "\n" + summary_code,
    encoding="utf-8",
)

importlib.reload(data)

summary = data.build_summary(app_profiles, app_suggestions)

# Small example: aggregate conversion must be 10/100, not
# the unweighted mean of the individual rates (50%).
example = pd.DataFrame({
    "ward_num": ["example_a", "example_b"],
    "ward_searches": [10, 90],
    "ward_bookings": [10, 0],
    "ward_completed_rides": [5, 0],
})
empty_suggestions = app_suggestions.iloc[:0]
example_summary = data.build_summary(example, empty_suggestions)

assert example_summary["search_to_booking_rate"] == 0.1
assert example_summary["booking_completion_rate"] == 0.5

empty_summary = data.build_summary(example.iloc[:0], empty_suggestions)
assert empty_summary["search_to_booking_rate"] is None
assert empty_summary["booking_completion_rate"] is None

import json
print("Aggregate-rate and zero-denominator checks passed.")
print("Summary includes all records, including w_0.")
print(json.dumps(summary, indent=2, allow_nan=False))

### Dashboard Setup
Installing Streamlit to build the interactive dashboard. The dashboard
will use the shared data-loading and summary functions.

In [ ]:
%pip -q install streamlit

import streamlit
print("Streamlit version:", streamlit.__version__)

### Create the Dashboard
Building a Streamlit interface using the shared data module.
Display aggregate metrics, individual record profiles, and investigation
suggestions while keeping data limitations visible.

In [ ]:
dashboard_code = 'import pandas as pd\nimport streamlit as st\nfrom nammaflow.data import load_data, build_summary\n\nst.set_page_config(page_title="Namma Yatri Flow", layout="wide")\n\nst.title("Namma Yatri Flow")\nst.caption("Bengaluru Mobility Analytics")\nst.caption("Independent analytics project using Namma Yatri data. Not affiliated with or endorsed by Namma Yatri.")\n\ntry:\n    profiles, suggestions = load_data()\nexcept (OSError, ValueError) as exc:\n    st.error(f"Unable to load validated reports: {exc}")\n    st.stop()\n\nst.warning(\n    "Area names use Namma Yatri’s published dashboard lookup. "\n    "Map boundaries and historical boundary compatibility remain unverified. "\n    "Suggestions support investigation; they do not establish causes "\n    "or operational priorities."\n)\n\nunmapped = profiles["ward_name_status"].eq("unmapped")\nif unmapped.any():\n    st.warning(f"{int(unmapped.sum())} records have no publisher name; shown as Unknown area with their original ID.")\nst.caption("Name source: [Namma Yatri official dashboard](https://nammayatri.in/open/), retrieved 12 November 2026.")\n\noverview, records, investigations = st.tabs(\n    ["Overview", "Record explorer", "Investigation explorer"]\n)\n\ndef percentage(value):\n    return "Unavailable" if value is None or pd.isna(value) else f"{value:.2%}"\n\nwith overview:\n    summary = build_summary(profiles, suggestions)\n    st.caption("Includes all records, including Other Wards (w_0).")\n\n    columns = st.columns(4)\n    for column, label, key in zip(\n        columns,\n        ["Records", "Searches", "Bookings", "Completed rides"],\n        ["record_count", "searches", "bookings", "completed_rides"],\n    ):\n        column.metric(label, f"{summary[key]:,}")\n\n    columns = st.columns(3)\n    for column, label, key in zip(\n        columns,\n        ["Search to booking", "Booking completion", "Search to completion"],\n        ["search_to_booking_rate", "booking_completion_rate",\n         "search_to_completion_rate"],\n    ):\n        column.metric(label, percentage(summary[key]))\n\n    st.caption("Aggregate rates use summed counts, not averages of record rates.")\n    st.subheader("Investigation coverage")\n    st.write(\n        f"{summary[\'suggestion_count\']} suggestions across "\n        f"{summary[\'records_with_suggestions\']} records."\n    )\n    st.dataframe(\n        suggestions.groupby("rule_id").size().rename("Suggestion count"),\n        width="stretch",\n    )\n\nwith records:\n    labels = profiles.set_index("ward_num")["display_name"].to_dict()\n    selected = st.selectbox(\n        "Area / ward", sorted(labels, key=lambda key: (labels[key].casefold(), key)),\n        format_func=labels.get\n    )\n    record = profiles.loc[profiles["ward_num"] == selected]\n    row = record.iloc[0]\n    st.subheader(row["display_name"])\n\n    columns = st.columns(3)\n    columns[0].metric("Searches", f"{int(row[\'ward_searches\']):,}")\n    columns[1].metric("Bookings", f"{int(row[\'ward_bookings\']):,}")\n    columns[2].metric("Completed rides", f"{int(row[\'ward_completed_rides\']):,}")\n\n    st.write("Search to booking:", percentage(row["ward_search_to_booking_rate"]))\n    st.write("Booking completion:", percentage(row["ward_booking_completion_rate"]))\n\n    if row["ward_bookings"] == 0:\n        st.info("No bookings: booking completion rate is undefined.")\n    if selected == "w_0":\n        st.info("Namma Yatri labels w_0 as Other Wards. Its exact coverage is unspecified; it is not a single mapped ward.")\n\n    st.subheader("Exploratory model context")\n    st.caption(\n        "Clusters describe exploratory groupings. Higher anomaly scores mean "\n        "more unusual patterns, not greater severity. Missing values are unavailable."\n    )\n    context = [\n        "has_model_results", "exclusion_reason", "exploratory_cluster",\n        "anomaly_score",\n    ]\n    st.dataframe(record[context], hide_index=True, width="stretch")\n\n    st.subheader("Suggestions for this record")\n    matching = suggestions.loc[suggestions["ward_num"] == selected]\n    if matching.empty:\n        st.info("No rules triggered. This does not establish that no issues exist.")\n    else:\n        st.dataframe(\n            matching[["ward_name", "ward_num", "rule_id", "suggested_investigation", "limitations"]],\n            hide_index=True, width="stretch",\n        )\n\n    with st.expander("All profile fields — raw rates are fractions"):\n        st.caption("geographic_mapping_status is the original report assessment. ward_name_status describes the newly added publisher label; boundaries remain unverified.")\n        st.dataframe(\n            record.T.rename(columns={record.index[0]: "Value"}).astype("string"),\n            width="stretch",\n        )\n\nwith investigations:\n    rule = st.selectbox(\n        "Rule", ["All"] + sorted(suggestions["rule_id"].unique().tolist())\n    )\n    filtered = suggestions if rule == "All" else suggestions[\n        suggestions["rule_id"] == rule\n    ]\n    st.write(\n        f"{len(filtered)} suggestions across "\n        f"{filtered[\'ward_num\'].nunique()} records."\n    )\n    st.caption("Raw rates are fractions. Missing values are unavailable. Original investigation text is retained; geographic caveats predate the added publisher names.")\n    st.dataframe(filtered, hide_index=True, width="stretch")\n\nwith st.expander("Data and interpretation limitations"):\n    st.markdown(\n        "- Historical reports do not establish forecasting accuracy.\\n"\n        "- Names come from Namma Yatri’s current dashboard lookup, retrieved 12 November 2026.\\n"\n        "- Names do not certify current legal wards or historical boundary compatibility; maps are deferred.\\n"\n        "- Original report geography fields and investigation text retain their historical assessment.\\n"\n        "- Supply timing compatibility remains unverified.\\n"\n        "- Rule thresholds are exploratory and sensitive to configuration.\\n"\n        "- Low booking counts can produce unstable rates.\\n"\n        "- Missing model outputs and undefined rates remain unavailable."\n    )\n'
dashboard_path = repo / "streamlit_app.py"
dashboard_path.write_text(dashboard_code)


### Dashboard Execution Checks
Test initial loading, overview metrics, zero-booking records,
w_0 context, and rule filtering before opening the dashboard.
These checks verify behavior; visual layout will be reviewed separately.

In [ ]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(str(dashboard_path), default_timeout=30).run()

def check_execution():
    assert not at.exception, [item.message for item in at.exception]
    assert not at.error, [item.value for item in at.error]

check_execution()

# Verify displayed overview totals.
# The first seven metrics belong to the Overview tab.
metrics = {item.label: item.value for item in at.metric[:7]}
assert metrics["Records"] == "244"
assert metrics["Searches"] == "36,112"
assert metrics["Bookings"] == "9,936"
assert metrics["Completed rides"] == "4,778"
print("PASS: Initial loading and overview totals")

# All identifiers must remain selectable.
assert set(at.selectbox[0].options) == set(app_profiles["display_name"])

for ward in ["b_168", "b_26", "b_44"]:
    at.selectbox[0].select(ward).run()
    check_execution()
    assert any("No bookings" in item.value for item in at.info)
    assert any(
        "Booking completion: Unavailable" in item.value
        for item in at.markdown
    )
print("PASS: Zero-booking records show unavailable completion rates")

at.selectbox[0].select("w_0").run()
check_execution()
assert any("Other Wards" in item.value for item in at.info)
print("PASS: w_0 remains selectable with context")

# The investigation table is the final dataframe in this dashboard.
for rule, expected in {"R01": 2, "R02": 76, "R03": 70, "R04": 1}.items():
    at.selectbox[1].select(rule).run()
    check_execution()
    table = at.dataframe[-1].value
    assert len(table) == expected
    assert set(table["rule_id"]) == {rule}
print("PASS: All four rule filters return the expected suggestions")

print("\nDashboard execution checks passed.")

In [ ]:
import subprocess
import sys
import time
import urllib.request
from google.colab import output

# Reuse our server if this cell is run again.
if "dashboard_process" not in globals() or dashboard_process.poll() is not None:
    dashboard_log = repo / "dashboard_preview.log"

    with dashboard_log.open("w") as log:
        dashboard_process = subprocess.Popen(
            [
                sys.executable, "-m", "streamlit", "run",
                str(dashboard_path),
                "--server.port=8501",
                "--server.address=0.0.0.0",
                "--server.headless=true",
                "--browser.gatherUsageStats=false",
            ],
            cwd=str(repo),
            stdout=log,
            stderr=subprocess.STDOUT,
        )

ready = False
for _ in range(30):
    if dashboard_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8501/_stcore/health", timeout=1
        ) as response:
            ready = response.status == 200
        if ready:
            break
    except OSError:
        pass
    time.sleep(0.5)

if ready:
    print("Dashboard server is ready. Preview below:")
    output.serve_kernel_port_as_iframe(8501, height=850)
else:
    print(dashboard_log.read_text())
    raise RuntimeError("Dashboard did not start; see the log above.")

In [ ]:
print("Server running:", dashboard_process.poll() is None)
print(dashboard_log.read_text()[-6000:])

In [ ]:
import json
from urllib.parse import urlsplit

# Get this session's Colab preview origin.
preview_url = output.eval_js("google.colab.kernel.proxyPort(8501)")
parsed = urlsplit(preview_url)
preview_origin = f"{parsed.scheme}://{parsed.netloc}"

if dashboard_process.poll() is None:
    dashboard_process.terminate()
    dashboard_process.wait(timeout=10)

with dashboard_log.open("w") as log:
    dashboard_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run",
            str(dashboard_path),
            "--server.port=8501",
            "--server.address=0.0.0.0",
            "--server.headless=true",
            "--browser.gatherUsageStats=false",
            "--server.corsAllowedOrigins=" + json.dumps([preview_origin]),
        ],
        cwd=str(repo),
        stdout=log,
        stderr=subprocess.STDOUT,
    )

ready = False
for _ in range(30):
    if dashboard_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8501/_stcore/health", timeout=1
        ) as response:
            ready = response.status == 200
        if ready:
            break
    except OSError:
        pass
    time.sleep(0.5)

if ready:
    print("Restarted with the Colab preview origin allowed.")
    output.serve_kernel_port_as_iframe(8501, height=850)
else:
    print(dashboard_log.read_text())
    raise RuntimeError("Dashboard restart failed.")

**Running it locally**

In [ ]:
from zipfile import ZipFile
from google.colab import files

package = repo.parent / "NammaFlow_dashboard_local.zip"

with ZipFile(package, "w") as archive:
    for relative in [
        "streamlit_app.py",
        "nammaflow/__init__.py",
    "data/reference/ward_names.json",
        "nammaflow/data.py",
        "reports/ward_level_intelligence/ward_profiles.csv",
        "reports/decision_recommendation/investigation_suggestions.csv",
    ]:
        archive.write(repo / relative, arcname=relative)

files.download(str(package))

In [ ]:
from pathlib import Path

ROOT = Path("/content/NammaFlow_phase9")

required = [
    "nammaflow/data.py",
    "streamlit_app.py",
    "reports/ward_level_intelligence/ward_profiles.csv",
    "reports/decision_recommendation/investigation_suggestions.csv",
]

for filename in required:
    path = ROOT / filename
    print(f"{'✅' if path.is_file() else '❌'} {filename}")

## FastAPI API
Expose the validated mobility reports through health, summary,
record, and investigation endpoints, reusing `nammaflow/data.py`.
Missing values will remain unavailable rather than being replaced with zero.

In [ ]:
%pip -q install fastapi uvicorn httpx

import fastapi
import uvicorn
import httpx

print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("HTTPX:", httpx.__version__)

In [ ]:
from pathlib import Path
ROOT = Path("/content/NammaFlow_phase9")
api_code = '\nimport json\nfrom contextlib import asynccontextmanager\n\nfrom fastapi import FastAPI, HTTPException, Query, Request\n\nfrom nammaflow.data import load_data, build_summary\n\n\ndef as_records(frame):\n    """Convert report rows to JSON-safe values, preserving missingness."""\n    return json.loads(frame.to_json(orient="records"))\n\n\n@asynccontextmanager\nasync def lifespan(app):\n    profiles, suggestions = load_data()\n    app.state.profiles = profiles\n    app.state.suggestions = suggestions\n    yield\n\n\napp = FastAPI(\n    title="Namma Yatri Flow API",\n    description=(\n        "Historical Bengaluru mobility analytics. "\n        "Independent project using Namma Yatri data; "\n        "not affiliated with or endorsed by Namma Yatri. "\n        "Area names use the publisher dashboard lookup; map boundaries and historical "\n        "boundary compatibility remain unverified. Original report geography fields "\n        "and investigation text retain their historical assessment. Investigation suggestions "\n        "are exploratory, not confirmed causes or severity rankings. "\n        "Summary rates are fractions; report fields retain source units."\n    ),\n    version="0.2.0",\n    lifespan=lifespan,\n)\n\n\n@app.get("/health")\ndef health(request: Request):\n    return {\n        "status": "ok",\n        "record_count": len(request.app.state.profiles),\n    }\n\n\n@app.get("/summary")\ndef summary(request: Request):\n    return build_summary(\n        request.app.state.profiles,\n        request.app.state.suggestions,\n    )\n\n\n@app.get("/records")\ndef records(\n    request: Request,\n    offset: int = Query(default=0, ge=0),\n    limit: int = Query(default=50, ge=1, le=250),\n):\n    profiles = request.app.state.profiles\n    ordered = profiles.sort_values("ward_num")\n    return {\n        "total": len(ordered),\n        "offset": offset,\n        "limit": limit,\n        "items": as_records(ordered.iloc[offset:offset + limit]),\n    }\n\n\n@app.get("/records/{record_id}")\ndef record_detail(record_id: str, request: Request):\n    profiles = request.app.state.profiles\n    matched = profiles[profiles["ward_num"].eq(record_id)]\n\n    if matched.empty:\n        raise HTTPException(status_code=404, detail="Record not found")\n\n    suggestions = request.app.state.suggestions\n    relevant = suggestions[suggestions["ward_num"].eq(record_id)]\n\n    return {\n        "profile": as_records(matched)[0],\n        "suggestions": as_records(relevant),\n    }\n\n\n@app.get("/investigations")\ndef investigations(\n    request: Request,\n    record_id: str | None = None,\n    rule_id: str | None = None,\n    offset: int = Query(default=0, ge=0),\n    limit: int = Query(default=50, ge=1, le=250),\n):\n    profiles = request.app.state.profiles\n    suggestions = request.app.state.suggestions\n\n    if record_id is not None:\n        if not profiles["ward_num"].eq(record_id).any():\n            raise HTTPException(status_code=404, detail="Record not found")\n        suggestions = suggestions[\n            suggestions["ward_num"].eq(record_id)\n        ]\n\n    if rule_id is not None:\n        known_rules = request.app.state.suggestions["rule_id"]\n        if not known_rules.eq(rule_id).any():\n            raise HTTPException(status_code=404, detail="Rule not found")\n        suggestions = suggestions[suggestions["rule_id"].eq(rule_id)]\n\n    ordered = suggestions.sort_values(["ward_num", "rule_id"])\n    return {\n        "total": len(ordered),\n        "offset": offset,\n        "limit": limit,\n        "items": as_records(ordered.iloc[offset:offset + limit]),\n    }\n'
(ROOT / "api.py").write_text(api_code)


In [ ]:
import sys
import importlib
import math

sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()

import api
api = importlib.reload(api)

from fastapi.testclient import TestClient

with TestClient(api.app) as client:
    def get_ok(path, **params):
        response = client.get(path, params=params)
        assert response.status_code == 200, response.text
        return response.json()

    # Startup and aggregate totals
    health = get_ok("/health")
    assert health == {"status": "ok", "record_count": 244}

    summary = get_ok("/summary")
    expected = {
        "record_count": 244,
        "searches": 36112,
        "bookings": 9936,
        "completed_rides": 4778,
        "suggestion_count": 149,
        "records_with_suggestions": 147,
        "suggestions_by_rule": {
            "R01": 2, "R02": 76, "R03": 70, "R04": 1
        },
    }
    for key, value in expected.items():
        assert summary[key] == value, (key, summary[key])

    for key, numerator, denominator in [
        ("search_to_booking_rate", 9936, 36112),
        ("booking_completion_rate", 4778, 9936),
        ("search_to_completion_rate", 4778, 36112),
    ]:
        assert math.isclose(
            summary[key], numerator / denominator, rel_tol=1e-12
        )

    print("✅ Health, totals, and aggregate rates")

    # Pagination must preserve every record exactly once
    rows = []
    for offset in range(0, 244, 50):
        page = get_ok("/records", offset=offset, limit=50)
        assert page["total"] == 244
        rows.extend(page["items"])

    ids = [row["ward_num"] for row in rows]
    assert len(ids) == len(set(ids)) == 244
    assert "w_0" in ids
    assert get_ok("/records", offset=244)["items"] == []

    # Every record detail must agree with its list entry
    for row in rows:
        detail = get_ok(f"/records/{row['ward_num']}")
        assert detail["profile"] == row
        assert all(
            item["ward_num"] == row["ward_num"]
            for item in detail["suggestions"]
        )

    print("✅ Pagination and all 244 record details")

    # Zero bookings must retain an undefined completion rate
    for record_id in ["b_168", "b_26", "b_44"]:
        profile = get_ok(f"/records/{record_id}")["profile"]
        assert profile["ward_bookings"] == 0
        assert profile["ward_booking_completion_rate"] is None
        assert profile["has_model_results"] is False

    profile = get_ok("/records/b_168")["profile"]
    assert profile["ward_searches"] == 13
    assert profile["ward_completed_rides"] == 0

    print("✅ Zero-booking records preserve JSON null")

    # Investigation filters and valid empty results
    all_suggestions = get_ok("/investigations", limit=250)
    assert all_suggestions["total"] == 149
    assert len(all_suggestions["items"]) == 149

    r01 = get_ok("/investigations", rule_id="R01")
    assert r01["total"] == 2
    assert {x["ward_num"] for x in r01["items"]} == {
        "b_131", "b_181"
    }

    combined = get_ok(
        "/investigations", record_id="b_181", rule_id="R01"
    )
    assert combined["total"] == 1
    assert combined["items"][0]["ward_num"] == "b_181"
    assert combined["items"][0]["rule_id"] == "R01"

    empty = get_ok("/investigations", record_id="b_1")
    assert empty["total"] == 0
    assert empty["items"] == []

    print("✅ Investigation filters and valid empty results")

    # Unknown identifiers and invalid pagination
    for path, params, status in [
        ("/records/unknown_record", {}, 404),
        ("/investigations", {"record_id": "unknown_record"}, 404),
        ("/investigations", {"rule_id": "unknown_rule"}, 404),
        ("/records", {"offset": -1}, 422),
        ("/records", {"limit": 0}, 422),
        ("/records", {"limit": 251}, 422),
        ("/investigations", {"offset": -1}, 422),
        ("/investigations", {"limit": 0}, 422),
        ("/investigations", {"limit": 251}, 422),
    ]:
        response = client.get(path, params=params)
        assert response.status_code == status, (
            path, params, response.status_code, response.text
        )

    print("✅ Unknown identifiers and invalid pagination")

print("\n✅ API integration checks passed.")

### API validation
FastAPI TestClient checks passed for startup, aggregate totals and
rates, all 244 record details, pagination, zero-booking null values,
investigation filters, empty results, and 404/422 responses.

These checks exercise the application in-process. A live Uvicorn
server check and deployment verification remain pending.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

paths = [
    "api.py",
    "streamlit_app.py",
    "nammaflow/__init__.py",
    "data/reference/ward_names.json",
    "nammaflow/data.py",
    "reports/ward_level_intelligence/ward_profiles.csv",
    "reports/decision_recommendation/investigation_suggestions.csv",
]

for name in paths:
    assert (ROOT / name).is_file(), f"Missing: {name}"

archive = ROOT.parent / "NammaFlow_phase9_current.zip"

with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in paths:
        z.write(ROOT / name, arcname=name)

files.download(str(archive))

### Local API verification
Installed the application requirements in a fresh Python 3.13
virtual environment on macOS. All API integration checks passed.

Uvicorn started successfully. Browser checks confirmed:
- /health returned HTTP 200 with 244 records.
- /docs displayed all five API endpoints.
- /summary returned HTTP 200 with the expected totals and rates.

A TestClient dependency deprecation warning appeared; checks passed.
Public deployment remains pending.